Defining function

In [4]:
def tiff_to_parquet(file_name, input_folder, output_folder):
    import os
    import tifffile as tiff
    import numpy as np
    import geopandas as gpd
    from shapely.geometry import shape
    from rasterio.features import shapes
    import pandas as pd
    from collections import defaultdict
    from tqdm import tqdm
    from pathlib import Path

    tiff_path = os.path.join(input_folder, file_name)
    output_name = file_name.split('_DAPI')[0]
    output_dir = Path(output_folder) / output_name
    output_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = output_dir / "nucleus_boundaries.parquet"

    print("Loading TIFF...")
    mask = tiff.imread(tiff_path)
    print('tiff loaded successfully')

    assert mask.ndim == 2, "Only 2D masks supported"
    assert mask.dtype.kind in ("i", "u"), "Mask must be integer-labeled"

    print("Extracting polygons...")

    # Collect all geometry parts per label
    geom_parts = defaultdict(list)
    num_shapes = len(np.unique(mask)) - 1

    for geom, value in tqdm(shapes(mask, mask=mask > 0), total=num_shapes, desc="Extracting shapes"):   #iterates through connected regions of same pixel values
        entity_id = int(value)
        poly = shape(geom) #shape(geom) doslovce izbaci polygon, us mislu onaj oblk stnaice 
        if poly.is_empty:
            continue

        # Normalize to polygons
        if poly.geom_type == "Polygon":
            geom_parts[entity_id].append(poly)
        elif poly.geom_type == "MultiPolygon":
            geom_parts[entity_id].extend(list(poly.geoms))
        
        #print(f"Processed EntityID: {entity_id}")

    print(f"Found {len(geom_parts)} unique nuclei")
    records = []

    '''
    #this is in case we want Multipolygon
    for entity_id, polygons in tqdm(geom_parts.items(), desc="Iterating through nuclei"):  #iterates through each unique label/cell
        multipolygon = MultiPolygon(polygons)
        centroid = multipolygon.centroid

        records.append({
            "EntityID": entity_id,
            "Geometry": multipolygon,
            "centroid_x": centroid.x,
            "centroid_y": centroid.y
            })
    '''

    for entity_id, polygons in tqdm(geom_parts.items(), desc="Iterating through nuclei"):
        for poly in polygons:
            centroid = poly.centroid

            records.append({
                "EntityID": entity_id,
                "geometry": poly,
                "centroid_x": centroid.x,
                "centroid_y": centroid.y
            })

    # Build GeoDataFrame
    gdf = gpd.GeoDataFrame(records, geometry="geometry",crs=None )

    # Enforce column order
    gdf = gdf[[
        "geometry",
        "EntityID",
        "centroid_x",
        "centroid_y"
        ]]

    # Removing multiple polygons per cell by keeping only the largest one
    gdf["area"] = gdf.geometry.area
    # Keep only the largest polygon per entity_id
    gdf_clean = gdf.loc[gdf.groupby("EntityID")["area"].idxmax()].copy()
    # drop helper column
    gdf_clean = gdf_clean.drop(columns="area")

    print(f"Writing Parquet... {parquet_path}")
    
    gdf_clean.to_parquet(parquet_path, index=False)
    print("Done.")
     

Main script for conerting tif to parquet, Kod nas je centrala

In [5]:
import os
from pathlib import Path

# 1. Define your folder path
input_folder = "/home/davids/main/sds/sd22c003/mi_spatialomics_project/MI_ApoE/Molecular_Cartography/cpsam_segmentations" # Replace with your actual path
output_folder = "/home/davids/main/projects/Segger_final/input_for_segger" # Replace with your desired output path

# 2. Get all .tif files
# This creates a list of all files ending in .tif
tif_files = [f for f in os.listdir(input_folder) if f.endswith(".tif")]

# 3. Loop through and extract the sample name
for file_name in tif_files:
    tiff_to_parquet(file_name, input_folder, output_folder)

Loading TIFF...
tiff loaded successfully
Extracting polygons...


Extracting shapes: 19294it [00:04, 3863.97it/s]                            


Found 19231 unique nuclei


Iterating through nuclei: 100%|██████████| 19231/19231 [00:00<00:00, 90859.14it/s]


Writing Parquet... /home/davids/main/projects/Segger_final/input_for_segger/33156-Slide-22_B1-1/nucleus_boundaries.parquet
Done.
Loading TIFF...
tiff loaded successfully
Extracting polygons...


...

Writing Parquet... /home/davids/main/projects/Segger_final/input_for_segger/33156-Slide-1_C2-1/nucleus_boundaries.parquet
Done.
Loading TIFF...
tiff loaded successfully
Extracting polygons...


Extracting shapes: 27892it [00:06, 4434.62it/s]                            


Found 27716 unique nuclei


Iterating through nuclei: 100%|██████████| 27716/27716 [00:00<00:00, 89757.22it/s]


Writing Parquet... /home/davids/main/projects/Segger_final/input_for_segger/33156-Slide-22_D1-1/nucleus_boundaries.parquet
Done.


In [ ]:
tiff_to_parquet('/home/davids/main/sds/sd22c003/mi_spatialomics_project/MI_ApoE/Molecular_Cartography/cpsam_segmentations/33156-Slide-1_A1-1_DAPI_cp_masks.tif', "/home/davids/main/projects/Segger_final/scripts/trash")

ValueError: '/home/davids/main/sds/sd22c003/mi_spatialomics_project/MI_ApoE/Molecular_Cartography/cpsam_segmentations/33156-Slide-1_A1-1_DAPI_cp_masks.tif' is not in the subpath of '$HOME/sds-hd' OR one path is relative and the other is absolute.

In [ ]:
#run tiff_to_parquet for all tiffs in the folder

tiff_to_parquet('', '/home/davids/main/projects/Segger_final/input_for_segger')

In [4]:
import tifffile as tiff
mask = tiff.imread('/home/davids/main/sds/sd22c003/mi_spatialomics_project/MI_ApoE/Molecular_Cartography/33156-Slide-1_submission/33156-Slide-1_A1-1_DAPI.tiff')

In [5]:
import numpy as np
num_shapes = len(np.unique(mask)) - 1